In [16]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

In [17]:

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

from sklearn.metrics import ConfusionMatrixDisplay

from sklearn.neighbors import KNeighborsClassifier

from sklearn.ensemble import RandomForestClassifier, VotingClassifier

from sklearn.linear_model import LogisticRegression

In [18]:

df = pd.read_csv("diabetes.csv")
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [19]:

X = df.drop(columns=['Outcome'])

y = df['Outcome']

In [20]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

In [21]:

pipe_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier())
])

param_knn = {
    'model__n_neighbors': np.arange(1, 26),
    'model__weights': ['uniform', 'distance'],
    'model__metric': ['euclidean', 'manhattan', 'minkowski']
}

knn_gs = GridSearchCV(pipe_knn, param_knn, cv=5, scoring='accuracy')

knn_gs.fit(X_train, y_train)

knn_best = knn_gs.best_estimator_

print("Best KNN params:", knn_gs.best_params_)

Best KNN params: {'model__metric': 'manhattan', 'model__n_neighbors': np.int64(19), 'model__weights': 'uniform'}


In [22]:

pipe_rf = Pipeline([
    ('model', RandomForestClassifier(random_state=42))
])

param_rf = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [None, 5, 10, 15],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4]
}

rf_gs = GridSearchCV(pipe_rf, param_rf, cv=5, scoring='accuracy')

rf_gs.fit(X_train, y_train)

rf_best = rf_gs.best_estimator_

print("Best RF params:", rf_gs.best_params_)

Best RF params: {'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__min_samples_split': 5, 'model__n_estimators': 50}


In [ ]:

pipe_log = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

param_log = [
    {'model__penalty': ['l1'], 'model__solver': ['liblinear', 'saga'], 'model__C': [0.01, 0.1, 1, 10]},
    {'model__penalty': ['l2'], 'model__solver': ['lbfgs', 'liblinear', 'saga'], 'model__C': [0.01, 0.1, 1, 10]},
    {'model__penalty': ['elasticnet'], 'model__solver': ['saga'], 'model__C': [0.01, 0.1, 1], 'model__l1_ratio': [0.5]},
    {'model__penalty': ['none'], 'model__solver': ['lbfgs', 'saga']}
]

log_gs = GridSearchCV(pipe_log, param_log, cv=5, scoring='accuracy')

log_gs.fit(X_train, y_train)

log_best = log_gs.best_estimator_

print("Best Logistic Regression params:", log_gs.best_params_)

In [ ]:

models = {
    "KNN": knn_best,
    "Random Forest": rf_best,
    "Logistic Regression": log_best
}

for name, model in models.items():
    score = model.score(X_test, y_test)
    print(f"{name} Accuracy: {score:.4f}")

In [ ]:

ensemble = VotingClassifier(
    estimators=[
        ('knn', knn_best),
        ('rf', rf_best),
        ('log', log_best)
    ],
    voting='soft'
)

ensemble.fit(X_train, y_train)

print("Ensemble Accuracy:", ensemble.score(X_test, y_test))

In [ ]:

y_pred = ensemble.predict(X_test)

print(classification_report(y_test, y_pred))

In [ ]:

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")

plt.title("Ensemble Confusion Matrix")
plt.show()

In [ ]:

plt.figure(figsize=(10, 7))

for name, model in models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.2f})")

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.show()

In [ ]:

y_proba = ensemble.predict_proba(X_test)[:, 1]

fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"Ensemble (AUC={roc_auc:.2f})", color='darkorange')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Ensemble ROC Curve")
plt.legend()
plt.show()

In [ ]:
import seaborn as sns

rf_model = rf_best.named_steps['model']

feat_importance = pd.Series(rf_model.feature_importances_, index=X.columns)
feat_importance = feat_importance.sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=feat_importance.values, y=feat_importance.index)
plt.title("Random Forest Feature Importance")
plt.show()